In [ ]:
import sys
dbutils.widgets.text("shared_lib_path", "/Workspace/Shared")
sys.path.insert(0, dbutils.widgets.get("shared_lib_path"))

dbutils.widgets.text("catalog", "vinoworld")
CATALOG   = dbutils.widgets.get("catalog")
BRONZE    = f"{CATALOG}.bronze"
SILVER    = f"{CATALOG}.silver"
GOLD      = f"{CATALOG}.gold"
AUDIT     = f"{CATALOG}.audit"
REPORTING = f"{CATALOG}.reporting"

# Empty string = no MANAGED LOCATION clause (Free Edition).
# Azure targets pass the ADLS abfss:// path via the bundle job parameter.
dbutils.widgets.text("managed_location", "")
MANAGED_LOCATION = dbutils.widgets.get("managed_location") or None

## catalog_ddl
Provisions the catalog, schemas, volumes, and all managed tables across the audit, bronze, silver, and gold layers, plus diagnostic and reporting views.
Safe to re-run — all statements use `CREATE IF NOT EXISTS` or `CREATE OR REPLACE`.

In [ ]:
from catalog_setup import (
    create_catalog,
    create_schemas,
    create_volume_schema,
    create_volumes,
    create_audit_tables,
    create_bronze_tables,
    create_silver_tables,
    create_gold_tables,
    create_audit_views,
    create_reporting_views,
)

In [ ]:
# Cell 3 — Catalog
result = create_catalog(spark, CATALOG, managed_location=MANAGED_LOCATION)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 4 — Medallion schemas and volume schema
result = create_schemas(spark, [BRONZE, SILVER, GOLD, AUDIT, REPORTING])
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

result = create_volume_schema(spark, CATALOG)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 5 — Volumes
# Add a dict here for each new source system volume.
VOLUME_DEFINITIONS = [
    {"name": "arancione",   "needs_archive": True},
    {"name": "celeste",     "needs_archive": True},
    {"name": "verde",       "needs_archive": True},
    {"name": "productdata", "needs_archive": True},
    {"name": "masterdata",  "needs_archive": False},
]

result = create_volumes(spark, dbutils, CATALOG, VOLUME_DEFINITIONS)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 6 — Audit tables
result = create_audit_tables(spark, AUDIT)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 7 — Bronze tables
result = create_bronze_tables(spark, BRONZE)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 8 — Silver tables
result = create_silver_tables(spark, SILVER)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 9 — Gold tables and views
result = create_gold_tables(spark, GOLD, SILVER)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 10 — Audit diagnostic views (depend on audit tables created in Cell 6)
result = create_audit_views(spark, AUDIT)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])

In [ ]:
# Cell 11 — Reporting views over gold.sales_fact (depend on gold + silver tables)
result = create_reporting_views(spark, REPORTING, SILVER, GOLD)
print(result["message"])
if result["status"] != "succeeded":
    raise RuntimeError(result["error"])